# 블로그 전처리 노트북

이 노트북은 `notebooks/00_crolling/blog_crolling.py`로 수집한 블로그 CSV를 01 단계에서 전처리합니다.
최종 목표는 블로그 데이터만 대상으로 날짜·section·댓글 구조를 표준화하고, Kiwi 명사 토큰 컬럼을 만들어 `data/blog_only/naver_blog_medical_quota_preprocessed.csv`와 `data/blog_only/naver_blog_medical_quota_preprocessed.pkl`로 저장하는 것입니다.

02 단계에서는 이 블로그 전처리 결과와 카페 전처리 결과를 합쳐 통합 데이터를 만듭니다.


## 0. 경로와 라이브러리 설정

노트북 실행 위치가 프로젝트 루트이든 `notebooks/01_preprocess`이든 같은 경로를 사용하도록 공통 경로 모듈을 불러옵니다.
블로그 입력 파일은 `data/blog_only/` 안에서 실제 존재하는 CSV 후보를 자동으로 선택합니다.


In [5]:
# 프로젝트 공통 경로를 설정합니다.
import json
import re
import sys
from pathlib import Path

import pandas as pd
from kiwipiepy import Kiwi
from tqdm import tqdm

_cwd = Path.cwd().resolve()
for _d in [_cwd, *_cwd.parents]:
    bootstrap_path = _d / "notebooks" / "lib" / "notebook_bootstrap.py"
    if bootstrap_path.is_file():
        sys.path.insert(0, str(_d / "notebooks" / "lib"))
        break
else:
    raise FileNotFoundError("notebooks/lib/notebook_bootstrap.py 을 찾을 수 없습니다. cwd=" + str(_cwd))

from notebook_bootstrap import setup_paths

PROJECT_ROOT = setup_paths()

from project_paths import DATA_BLOG_ONLY

BLOG_CSV_CANDIDATES = [
    DATA_BLOG_ONLY / "naver_blog_medical_quota.csv",
    DATA_BLOG_ONLY / "naver_blog_medical_quota_final.csv",
    DATA_BLOG_ONLY / "naver_blog_medical_quota_final_jupyter.csv",
    DATA_BLOG_ONLY / "naver_blog_medical_quota_preprocessed.csv",
]

OUTPUT_CSV = DATA_BLOG_ONLY / "naver_blog_medical_quota_preprocessed.csv"
OUTPUT_PKL = DATA_BLOG_ONLY / "naver_blog_medical_quota_preprocessed.pkl"

print("블로그 데이터 폴더:", DATA_BLOG_ONLY)

블로그 데이터 폴더: /Users/hyerimchoi/DJD/datacrolling/20241983최혜림_20241285이서연_드론영상비정형데이터AI분석_중간과제/20241983최혜림_20241285이서연_드론비정형데이터AI분석/data/blog_only


## 1. 블로그 CSV 로드와 표준 컬럼 정리

블로그 크롤링 CSV는 실행 시점에 따라 컬럼명이 조금 다를 수 있습니다.
아래 함수들은 제목, 본문, 좋아요, 댓글 수, 댓글 목록, 채널, 날짜, section을 통합 단계에서 쓰는 표준 컬럼으로 맞춥니다.


In [6]:
# CSV 인코딩과 컬럼 차이를 흡수하는 보조 함수입니다.
STANDARD_COLUMNS = ["title", "doc", "like", "comment_cnt", "comment_list", "ch", "date", "section"]


def first_existing(paths):
    """후보 경로 중 실제 존재하는 첫 파일을 반환합니다."""
    for path in paths:
        if path.is_file():
            return path
    raise FileNotFoundError("data/blog_only/에 블로그 CSV가 없습니다: " + ", ".join(p.name for p in paths))


def read_csv_with_fallback(path):
    """UTF-8, CP949 등 자주 쓰이는 인코딩을 순서대로 시도합니다."""
    last_error = None
    for encoding in ["utf-8-sig", "utf-8", "cp949", "euc-kr"]:
        try:
            return pd.read_csv(path, low_memory=False, encoding=encoding)
        except UnicodeDecodeError as exc:
            last_error = exc
    raise last_error


def to_int(value, default=0):
    """쉼표나 문자 단위가 섞인 숫자 값을 안전하게 정수로 바꿉니다."""
    if pd.isna(value):
        return default
    text = str(value).strip().replace(",", "")
    if not text:
        return default
    if text.endswith(".0"):
        text = text[:-2]
    try:
        return int(text)
    except ValueError:
        digits = "".join(ch for ch in text if ch.isdigit())
        return int(digits) if digits else default


def normalize_date(value):
    """여러 날짜 표기를 YYYY-MM-DD 문자열로 통일합니다."""
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if not text:
        return ""
    if text.endswith(".0"):
        text = text[:-2]
    parsed = pd.to_datetime(text, format="%Y%m%d", errors="coerce")
    if pd.isna(parsed):
        parsed = pd.to_datetime(text, errors="coerce")
    return parsed.strftime("%Y-%m-%d") if not pd.isna(parsed) else ""


def normalize_section(value, date_value=""):
    """명시된 section이 없으면 날짜 기준으로 1~4구간을 부여합니다."""
    section = to_int(value, default=0)
    if section in {1, 2, 3, 4}:
        return section
    normalized_date = normalize_date(date_value)
    if "2024-01-01" <= normalized_date <= "2024-03-31":
        return 1
    if "2024-04-01" <= normalized_date <= "2024-06-30":
        return 2
    if "2024-07-01" <= normalized_date <= "2024-12-31":
        return 3
    if "2025-01-01" <= normalized_date <= "2025-06-30":
        return 4
    return 0


def parse_comment_json(value):
    """이미 JSON 문자열 또는 list로 저장된 댓글을 list[dict]로 복원합니다."""
    if isinstance(value, list):
        return value
    if pd.isna(value) or not str(value).strip():
        return []
    try:
        parsed = json.loads(str(value))
    except json.JSONDecodeError:
        return []
    return parsed if isinstance(parsed, list) else []


def build_comment_list(row):
    """분리된 댓글 텍스트·날짜·좋아요 컬럼을 list[dict] 구조로 묶습니다."""
    comments = parse_comment_json(row.get("comments_json", ""))
    if comments:
        return [
            {
                "comment_content": str(item.get("text", item.get("comment_content", ""))).strip(),
                "comment_like": to_int(item.get("like_count", item.get("comment_like", 0))),
                "comment_date": normalize_date(item.get("time", item.get("comment_date", ""))),
            }
            for item in comments
        ]

    texts = [p.strip() for p in str(row.get("comments_text", "")).split(" || ") if p.strip()]
    times = [p.strip() for p in str(row.get("comment_times", "")).split(" || ") if p.strip()]
    likes = [p.strip() for p in str(row.get("comment_like_counts", "")).split(" || ") if p.strip()]
    out = []
    for idx in range(max(len(texts), len(times), len(likes))):
        out.append(
            {
                "comment_content": texts[idx] if idx < len(texts) else "",
                "comment_like": to_int(likes[idx], default=0) if idx < len(likes) else 0,
                "comment_date": normalize_date(times[idx]) if idx < len(times) else "",
            }
        )
    return out


def column_or_default(raw, column, default=""):
    """컬럼이 없을 때도 행 수에 맞는 기본 Series를 반환합니다."""
    if column in raw.columns:
        return raw[column]
    return pd.Series([default] * len(raw), index=raw.index)


def standardize_blog_dataframe(raw):
    """원본 또는 이미 정리된 블로그 CSV를 표준 컬럼 DataFrame으로 변환합니다."""
    raw = raw.copy()
    if set(STANDARD_COLUMNS).issubset(raw.columns):
        prepared = raw[STANDARD_COLUMNS].copy()
        prepared["comment_list"] = prepared["comment_list"].apply(parse_comment_json)
    else:
        like_col = "post_like_count" if "post_like_count" in raw.columns else "like_count"
        date_col = "date" if "date" in raw.columns else "post_date"
        prepared = pd.DataFrame(
            {
                "title": column_or_default(raw, "title").fillna(""),
                "doc": (column_or_default(raw, "content") if "content" in raw.columns else column_or_default(raw, "doc")).fillna(""),
                "like": column_or_default(raw, like_col, 0).apply(to_int),
                "comment_cnt": (column_or_default(raw, "comment_count", 0) if "comment_count" in raw.columns else column_or_default(raw, "comment_cnt", 0)).apply(to_int),
                "comment_list": raw.apply(build_comment_list, axis=1),
                "ch": "blog",
                "date": column_or_default(raw, date_col).apply(normalize_date),
                "section": raw.apply(lambda row: normalize_section(row.get("section", ""), row.get(date_col, "")), axis=1),
            }
        )
    prepared["like"] = prepared["like"].apply(to_int)
    prepared["comment_cnt"] = prepared["comment_cnt"].apply(to_int)
    prepared["date"] = prepared["date"].apply(normalize_date)
    prepared["section"] = prepared.apply(lambda row: normalize_section(row.get("section", ""), row.get("date", "")), axis=1)
    prepared["ch"] = "blog"
    prepared = prepared[prepared["section"].isin([1, 2, 3, 4])].reset_index(drop=True)
    return prepared[STANDARD_COLUMNS]

In [7]:
# 블로그 CSV를 불러와 표준 컬럼으로 정리합니다.
INPUT_CSV = first_existing(BLOG_CSV_CANDIDATES)
raw_blog = read_csv_with_fallback(INPUT_CSV)
blog_df = standardize_blog_dataframe(raw_blog)

print("입력 파일:", INPUT_CSV)
print("전처리 대상 행 수:", len(blog_df))
print(blog_df["section"].value_counts().sort_index())
blog_df.head()

입력 파일: /Users/hyerimchoi/DJD/datacrolling/20241983최혜림_20241285이서연_드론영상비정형데이터AI분석_중간과제/20241983최혜림_20241285이서연_드론비정형데이터AI분석/data/blog_only/naver_blog_medical_quota.csv
전처리 대상 행 수: 5354
section
1    1664
2    1789
3    1685
4     216
Name: count, dtype: int64


,title,doc,like,comment_cnt,comment_list,ch,date,section
0,"""병원 남은 참의사들"" 신상·조롱 퍼지자…일부 전공의 ""존중 해야""",의대정원 증원에 반발한 전공의들이 진료거부를 이어가고 있는 지난 5일 오전 서울시 ...,8,2,[{'comment_content': '입시도사님 :) 오늘도 방문했어요 ㅎㅎ 정성...,blog,2024-03-10,1
1,정부는 의사를 이길 수 없다....?,정부는 의사를 이길 수 없다....? #노환규 전 #대한의사협회 (의협) 회장이 경...,32,1,[{'comment_content': '생명은 누구에게나 소중한 것이다. 생명을 볼...,blog,2024-03-10,1
2,"[입시정보] [각 대학별 의대 증원 신청, 서울대 의대 180명~200명 예상 - ...",의대 증원 이슈가 마무리되지 않고 있습니다. 이러한 상황 속 정부에서는 각 대학의 ...,17,12,"[{'comment_content': '포스팅 잘봤어요 자주 방문할게요~ :)', ...",blog,2024-03-14,1
3,[속보] 정부 “일본 의대 증원 실패 후 정원 축소 사실 아냐… 교육 질 저하 등 ...,[속보] 정부 “일본 의대 증원 실패 후 정원 축소 사실 아냐… 교육 질 저하 등 ...,1,1,[{'comment_content': '공감 누르고 갑니다 답방 부탁 드릴께요 ^^...,blog,2024-03-14,1
4,"'의대 증원' 오늘 집행정지 소송 심문…교수들 ""입시요강 변경 불가""","전국 33개 의대 교수협의회 대표 ""명백한 사기행위"" 전공의·의대생도 행정소송 제기...",0,1,[{'comment_content': '의대 교수들의 결연한 의지에 박수를 보냅니다...,blog,2024-03-14,1


블로그 원본 CSV에서 5,354건이 전처리 대상으로 선택되었습니다. section 분포는 1구간과 3구간이 상대적으로 많고 4구간이 적은 것이 확인됩니다.

이 셀에서 `date`, `section`, `comment_list`, `ch=blog`가 표준화시켜 02 통합 단계에서 카페 데이터와 같은 구조로 결합할 예정입니다.

## 2. 텍스트 정리와 Kiwi 명사 추출

제목, 본문, 댓글 텍스트에서 HTML 잔여 기호와 불필요한 공백을 정리한 뒤 Kiwi로 일반명사(`NNG`)와 고유명사(`NNP`)만 추출합니다.
불용어 처리는 02 통합 단계에서 공통 기준으로 적용하므로, 여기서는 원천별 명사 토큰을 만드는 데 집중합니다.


In [8]:
# 텍스트 정리와 명사 추출 함수입니다.
kiwi = Kiwi()


def clean_text(value):
    """HTML 흔적, URL, 과도한 공백을 줄여 형태소 분석 입력을 안정화합니다."""
    if pd.isna(value):
        return ""
    text = str(value)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^0-9A-Za-z가-힣\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_nouns(text):
    """Kiwi 결과 중 2글자 이상 일반명사·고유명사만 남깁니다."""
    tokens = kiwi.tokenize(clean_text(text))
    return [token.form for token in tokens if token.tag in {"NNG", "NNP"} and len(token.form) > 1]


def comment_texts(comment_list):
    """댓글 list[dict]에서 댓글 본문만 추출합니다."""
    if not isinstance(comment_list, list):
        return []
    return [str(item.get("comment_content", "")) for item in comment_list if isinstance(item, dict)]


blog_df["title"] = blog_df["title"].apply(clean_text)
blog_df["doc"] = blog_df["doc"].apply(clean_text)
blog_df["comment_list"] = blog_df["comment_list"].apply(
    lambda rows: [
        {**item, "comment_content": clean_text(item.get("comment_content", ""))}
        for item in rows
        if isinstance(item, dict)
    ]
)

blog_df["title_token_noun"] = [extract_nouns(text) for text in tqdm(blog_df["title"], desc="제목 명사 추출")]
blog_df["document_token_noun"] = [extract_nouns(text) for text in tqdm(blog_df["doc"], desc="본문 명사 추출")]
blog_df["comment_token_noun"] = [
    [extract_nouns(text) for text in comment_texts(rows)]
    for rows in tqdm(blog_df["comment_list"], desc="댓글 명사 추출")
]

blog_df[["title", "title_token_noun", "document_token_noun", "comment_token_noun"]].head()

Quantization is not supported for ArchType::neon. Fall back to non-quantized model.
Quantization is not supported for ArchType::neon. Fall back to non-quantized model.
댓글 명사 추출: 100%|██████████| 5354/5354 [00:38<00:00, 139.98it/s]


,title,title_token_noun,document_token_noun,comment_token_noun
0,병원 남은 참의사들 신상 조롱 퍼지자 일부 전공의 존중 해야,"[병원, 의사, 신상, 조롱, 일부, 전공의, 존중]","[의대, 정원, 증원, 반발, 전공의, 진료, 거부, 오전, 서울시, 병원, 의료진...","[[입시, 도사, 오늘, 방문, 정성, 월요일, 방문], [전공의, 희생, 노력, ..."
1,정부는 의사를 이길 수 없다,"[정부, 의사]","[정부, 의사, 노환규, 대한의사협회, 의협, 회장, 경찰, 출석, 조사, 귀가, ...","[[생명, 생명, 볼모, 행동, 인륜]]"
2,입시정보 각 대학별 의대 증원 신청 서울대 의대 180명 200명 예상 베리타스알파,"[입시, 정보, 대학, 의대, 증원, 신청, 서울대, 의대, 예상, 베리타스알파]","[의대, 증원, 이슈, 마무리, 상황, 정부, 대학, 증원, 신청, 영향력, 서울대...","[[포스팅, 방문], [댓글, 감사, 소통], [의대, 증원, 시국, 해결], [동..."
3,속보 정부 일본 의대 증원 실패 후 정원 축소 사실 아냐 교육 질 저하 등 부작용 ...,"[속보, 정부, 일본, 의대, 증원, 실패, 정원, 축소, 사실, 교육, 저하, 부...","[속보, 정부, 일본, 의대, 증원, 실패, 정원, 축소, 사실, 교육, 저하, 부...","[[공감, 답방, 부탁]]"
4,의대 증원 오늘 집행정지 소송 심문 교수들 입시요강 변경 불가,"[의대, 증원, 집행, 정지, 소송, 심문, 교수, 입시, 요강, 변경, 불가]","[전국, 의대, 교수, 협의회, 대표, 사기, 행위, 전공, 의대, 행정, 소송, ...","[[의대, 교수, 의지, 박수, 응원]]"


## 3. 저장

02 통합 단계는 PKL을 우선 사용하고, 없을 때 CSV를 읽어 리스트를 복원합니다.


In [9]:
# 블로그 전처리 결과를 저장합니다.
OUTPUT_COLUMNS = STANDARD_COLUMNS + ["title_token_noun", "document_token_noun", "comment_token_noun"]
blog_out = blog_df[OUTPUT_COLUMNS].copy()

csv_out = blog_out.copy()
for col in ["comment_list", "title_token_noun", "document_token_noun", "comment_token_noun"]:
    csv_out[col] = csv_out[col].apply(lambda value: json.dumps(value, ensure_ascii=False))

csv_out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
blog_out.to_pickle(OUTPUT_PKL)

print("CSV 저장:", OUTPUT_CSV)
print("PKL 저장:", OUTPUT_PKL)
print("최종 행 수:", len(blog_out))
blog_out.head()

CSV 저장: /Users/hyerimchoi/DJD/datacrolling/20241983최혜림_20241285이서연_드론영상비정형데이터AI분석_중간과제/20241983최혜림_20241285이서연_드론비정형데이터AI분석/data/blog_only/naver_blog_medical_quota_preprocessed.csv
PKL 저장: /Users/hyerimchoi/DJD/datacrolling/20241983최혜림_20241285이서연_드론영상비정형데이터AI분석_중간과제/20241983최혜림_20241285이서연_드론비정형데이터AI분석/data/blog_only/naver_blog_medical_quota_preprocessed.pkl
최종 행 수: 5354


,title,doc,like,comment_cnt,comment_list,ch,date,section,title_token_noun,document_token_noun,comment_token_noun
0,병원 남은 참의사들 신상 조롱 퍼지자 일부 전공의 존중 해야,의대정원 증원에 반발한 전공의들이 진료거부를 이어가고 있는 지난 5일 오전 서울시 ...,8,2,[{'comment_content': '입시도사님 오늘도 방문했어요 정성스럽게 쓰신...,blog,2024-03-10,1,"[병원, 의사, 신상, 조롱, 일부, 전공의, 존중]","[의대, 정원, 증원, 반발, 전공의, 진료, 거부, 오전, 서울시, 병원, 의료진...","[[입시, 도사, 오늘, 방문, 정성, 월요일, 방문], [전공의, 희생, 노력, ..."
1,정부는 의사를 이길 수 없다,정부는 의사를 이길 수 없다 노환규 전 대한의사협회 의협 회장이 경찰에 출석해서 1...,32,1,[{'comment_content': '생명은 누구에게나 소중한 것이다 생명을 볼모...,blog,2024-03-10,1,"[정부, 의사]","[정부, 의사, 노환규, 대한의사협회, 의협, 회장, 경찰, 출석, 조사, 귀가, ...","[[생명, 생명, 볼모, 행동, 인륜]]"
2,입시정보 각 대학별 의대 증원 신청 서울대 의대 180명 200명 예상 베리타스알파,의대 증원 이슈가 마무리되지 않고 있습니다 이러한 상황 속 정부에서는 각 대학의 증...,17,12,"[{'comment_content': '포스팅 잘봤어요 자주 방문할게요', 'com...",blog,2024-03-14,1,"[입시, 정보, 대학, 의대, 증원, 신청, 서울대, 의대, 예상, 베리타스알파]","[의대, 증원, 이슈, 마무리, 상황, 정부, 대학, 증원, 신청, 영향력, 서울대...","[[포스팅, 방문], [댓글, 감사, 소통], [의대, 증원, 시국, 해결], [동..."
3,속보 정부 일본 의대 증원 실패 후 정원 축소 사실 아냐 교육 질 저하 등 부작용 ...,속보 정부 일본 의대 증원 실패 후 정원 축소 사실 아냐 교육 질 저하 등 부작용 ...,1,1,"[{'comment_content': '공감 누르고 갑니다 답방 부탁 드릴께요', ...",blog,2024-03-14,1,"[속보, 정부, 일본, 의대, 증원, 실패, 정원, 축소, 사실, 교육, 저하, 부...","[속보, 정부, 일본, 의대, 증원, 실패, 정원, 축소, 사실, 교육, 저하, 부...","[[공감, 답방, 부탁]]"
4,의대 증원 오늘 집행정지 소송 심문 교수들 입시요강 변경 불가,전국 33개 의대 교수협의회 대표 명백한 사기행위 전공의 의대생도 행정소송 제기 첫...,0,1,[{'comment_content': '의대 교수들의 결연한 의지에 박수를 보냅니다...,blog,2024-03-14,1,"[의대, 증원, 집행, 정지, 소송, 심문, 교수, 입시, 요강, 변경, 불가]","[전국, 의대, 교수, 협의회, 대표, 사기, 행위, 전공, 의대, 행정, 소송, ...","[[의대, 교수, 의지, 박수, 응원]]"


## 저장물 검증

전처리 결과를 저장한 뒤 바로 다시 읽어 행 수, 필수 컬럼, section 분포, 빈 토큰 문서 수를 확인합니다. 이 검증은 02 통합 단계에서 경로 오류나 컬럼 누락이 발생하지 않도록 하기 위한 마지막 점검입니다.

In [10]:
# 저장된 PKL을 다시 읽어 02 통합 단계 입력으로 문제가 없는지 검증합니다.
validated_blog = pd.read_pickle(OUTPUT_PKL)
required_cols = OUTPUT_COLUMNS
missing_cols = [col for col in required_cols if col not in validated_blog.columns]
if missing_cols:
    raise KeyError(f"블로그 전처리 저장물 누락 컬럼: {missing_cols}")

empty_token_rows = validated_blog[
    validated_blog[["title_token_noun", "document_token_noun", "comment_token_noun"]]
    .applymap(lambda value: len(value) if isinstance(value, list) else 0)
    .sum(axis=1)
    == 0
]
duplicate_titles = validated_blog.duplicated(subset=["title", "date"], keep=False).sum()

print("저장물 행 수:", len(validated_blog))
print("section 분포:")
print(validated_blog["section"].value_counts().sort_index())
print("빈 토큰 문서 수:", len(empty_token_rows))
print("제목+날짜 기준 중복 의심 행 수:", duplicate_titles)
validated_blog.head(3)

저장물 행 수: 5354
section 분포:
section
1    1664
2    1789
3    1685
4     216
Name: count, dtype: int64
빈 토큰 문서 수: 0
제목+날짜 기준 중복 의심 행 수: 6


/var/folders/wg/2y3_zgls3jxc_lntn4_q23mm0000gn/T/ipykernel_61286/2110528688.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  .applymap(lambda value: len(value) if isinstance(value, list) else 0)


,title,doc,like,comment_cnt,comment_list,ch,date,section,title_token_noun,document_token_noun,comment_token_noun
0,병원 남은 참의사들 신상 조롱 퍼지자 일부 전공의 존중 해야,의대정원 증원에 반발한 전공의들이 진료거부를 이어가고 있는 지난 5일 오전 서울시 ...,8,2,[{'comment_content': '입시도사님 오늘도 방문했어요 정성스럽게 쓰신...,blog,2024-03-10,1,"[병원, 의사, 신상, 조롱, 일부, 전공의, 존중]","[의대, 정원, 증원, 반발, 전공의, 진료, 거부, 오전, 서울시, 병원, 의료진...","[[입시, 도사, 오늘, 방문, 정성, 월요일, 방문], [전공의, 희생, 노력, ..."
1,정부는 의사를 이길 수 없다,정부는 의사를 이길 수 없다 노환규 전 대한의사협회 의협 회장이 경찰에 출석해서 1...,32,1,[{'comment_content': '생명은 누구에게나 소중한 것이다 생명을 볼모...,blog,2024-03-10,1,"[정부, 의사]","[정부, 의사, 노환규, 대한의사협회, 의협, 회장, 경찰, 출석, 조사, 귀가, ...","[[생명, 생명, 볼모, 행동, 인륜]]"
2,입시정보 각 대학별 의대 증원 신청 서울대 의대 180명 200명 예상 베리타스알파,의대 증원 이슈가 마무리되지 않고 있습니다 이러한 상황 속 정부에서는 각 대학의 증...,17,12,"[{'comment_content': '포스팅 잘봤어요 자주 방문할게요', 'com...",blog,2024-03-14,1,"[입시, 정보, 대학, 의대, 증원, 신청, 서울대, 의대, 예상, 베리타스알파]","[의대, 증원, 이슈, 마무리, 상황, 정부, 대학, 증원, 신청, 영향력, 서울대...","[[포스팅, 방문], [댓글, 감사, 소통], [의대, 증원, 시국, 해결], [동..."
